In [2]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_example
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,44,False)#OUTPUT_DIM_NOTES,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
# cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch02_valAcc0.9551.weights.h5')#
cnn_model.load_weights('guitarmidi.weights.h5')

input_data_dir = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices'
input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/data_slices/training_subset/filtered_poly_data0.tfrecord'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_example).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_data_gen
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Initial input shape: (None, 148, 512)
After first Conv2D: (None, 74, 64)
Extracting string from filters 0 to 26
String 0 section shape: (None, 26, 64)
String 0 after first Conv1D: (None, 26, 128)
Extracting string from filters 11 to 36
String 11 section shape: (None, 25, 64)
String 11 after first Conv1D: (None, 25, 128)
Extracting string from filters 21 to 46
String 21 section shape: (None, 25, 64)
String 21 after first Conv1D: (None, 25, 128)
Extracting string from filters 31 to 56
String 31 section shape: (None, 25, 64)
String 31 after first Conv1D: (None, 25, 128)
Extracting string from filters 39 to 64
String 39 section shape: (None, 25, 64)
String 39 after first Conv1D: (None, 25, 128)
Extracting string from filters 49 to 73
String 49 section shape: (None, 24, 64)
String 49 after first Conv1D: (None, 24, 128)


Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_1       │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_3 (Reshape) │ (None, 148, 256,  │          0 │ input_layer_1[0]… │
│                     │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 148, 32,   │        272 │ reshape_3[0][0]   │
│                     │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 148, 32,   │         64 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_16      │ (None, 148, 32,   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_2 │ (None, 148, 32,   │          0 │ leaky_re_lu_16[0… │
│ (SpatialDropout2D)  │ 16)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_4 (Reshape) │ (None, 148, 512)  │          0 │ spatial_dropout2… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 148, 512)  │    263,040 │ reshape_4[0][0],  │
│ (MultiHeadAttentio… │                   │            │ reshape_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 148, 512)  │          0 │ reshape_4[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 148, 512)  │      1,024 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, 148, 256)  │    131,328 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, 148, 512)  │    131,584 │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 148, 512)  │          0 │ dense_4[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 148, 512)  │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_4[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 148, 512)  │      1,024 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 148, 32)   │    114,720 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 148, 32)   │        128 │ conv1d_14[0][0]   │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ leaky_re_lu_17      │ (None, 148, 32)   │          0 │ batch_normalizat… │
│ (LeakyReLU)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 4,553,244 (17.37 MB)

 Trainable params: 4,548,348 (17.35 MB)

...

INFO:tensorflow:Assets written to: /tmp/tmpx5_ncn9g/assets


Saved artifact at '/tmp/tmpx5_ncn9g'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='keras_tensor_112')
Output Type:
  TensorSpec(shape=(None, 44), dtype=tf.float32, name=None)
Captures:
  127039966281552: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553747856: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553747088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553746896: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553747664: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553747280: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553748048: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553749008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553748816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553749392: TensorSpec(shape=(), dtype=tf.resource, name=None)
  127040553

/home/gerald/anaconda3/envs/gpu/lib/python3.13/site-packages/tensorflow/lite/python/convert.py:846: UserWarning: Statistics for quantized inputs were expected, but not specified; continuing anyway.
  warnings.warn(
W0000 00:00:1773300268.661383  193833 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1773300268.661398  193833 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1773300268.661482  193833 reader.cc:83] Reading SavedModel from: /tmp/tmpx5_ncn9g
I0000 00:00:1773300268.662960  193833 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1773300268.662967  193833 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpx5_ncn9g
I0000 00:00:1773300268.682821  193833 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1773300268.809725  193833 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpx5_ncn9g
I0000 00:00:1773300268.840427  193833 loader.cc:471] SavedModel load for 

TFLite model saved as guitarmidi.tflite


fully_quantize: 0, inference_type: 6, input_inference_type: FLOAT32, output_inference_type: FLOAT32
W0000 00:00:1773300270.766979  193833 flatbuffer_export.cc:3851] Skipping runtime version metadata in the model. This will be generated by the exporter.
